# 🛒 Análise de Cesta de Compras — UK Online Retail
## Problema de Negócio
> **Objetivo:** Identificar quais produtos são comprados juntos para criar estratégias de *bundles* e *cross-selling* que aumentem o **AOV (Average Order Value — Ticket Médio)**.
>
> **Dataset:** UCI Online Retail — 541K transações, loja UK, 2010–2011.
>
> **Método:** Algoritmo Apriori + Regras de Associação (mlxtend) → Regras classificadas por **Lift**.

---
*Autor: [Seu Nome] | Data: Fevereiro 2026 | Stack: Python · pandas · mlxtend · matplotlib · seaborn*

## 0. Configuração e Importações

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
ACCENT = '#4361EE'

DATA_PATH        = 'data/online_retail.csv'
OUTPUT_RULES_CSV = 'output/association_rules.csv'
OUTPUT_ITEMS_CSV = 'output/frequent_items.csv'
MIN_SUPPORT      = 0.01   # Mínimo 1% das transações
MIN_CONFIDENCE   = 0.20   # Confiança mínima de 20%
MIN_LIFT         = 1.0    # Apenas regras com lift > 1 (correlação positiva)
TARGET_COUNTRY   = 'United Kingdom'

print('✅ Bibliotecas carregadas com sucesso.')

---
## 1. Carregamento de Dados & AED — Análise Exploratória dos Dados

### 💼 Comentário Executivo
> Antes de construir qualquer modelo, precisamos entender a **distribuição do negócio**: volume de transações, sazonalidade de vendas, produtos mais vendidos e receita por país.
>
> Esta etapa detecta **anomalias** (devoluções, preços negativos) que enviasariam o modelo e fornece contexto para interpretar as regras de associação como decisões reais de negócio.
>
> **Descoberta-chave:** O Reino Unido representa ~80% da receita — isolamos os dados do UK para uma análise de cesta mais focada e de alta qualidade.

In [ ]:
try:
    df = pd.read_csv(DATA_PATH, encoding='latin-1')
    print(f'✅ Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
except FileNotFoundError:
    import glob
    print(f'Arquivo não encontrado em {DATA_PATH}. Disponíveis: {glob.glob("**/*.csv", recursive=True)}')
    raise

df.head(3)

In [ ]:
display(df.dtypes.to_frame('Tipo'))
display(df[['Quantity', 'UnitPrice']].describe().round(2))

nulls = df.isnull().sum()
null_report = pd.DataFrame({'Nulos': nulls, '%': (nulls/len(df)*100).round(2)})
print('\nValores nulos:')
display(null_report[null_report['Nulos'] > 0])

In [ ]:
import os
os.makedirs('output', exist_ok=True)

country_revenue = (
    df.assign(Receita=lambda x: x['Quantity'] * x['UnitPrice'])
      .groupby('Country')['Receita'].sum().sort_values(ascending=False).head(10)
)

fig, ax = plt.subplots(figsize=(12, 5))
colors = [ACCENT if c == TARGET_COUNTRY else '#ADB5BD' for c in country_revenue.index[::-1]]
ax.barh(country_revenue.index[::-1], country_revenue.values[::-1], color=colors)
ax.set_title('Top 10 Países por Receita — UK domina o volume', fontsize=13, fontweight='bold')
ax.set_xlabel('Receita Total (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('output/01_revenue_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

uk_share = country_revenue[TARGET_COUNTRY] / country_revenue.sum() * 100
print(f'\n📌 UK representa {uk_share:.1f}% da receita total → Apriori focado apenas no UK.')

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['AnoMes']      = df['InvoiceDate'].dt.to_period('M')

receita_mensal = (
    df.assign(Receita=lambda x: x['Quantity'] * x['UnitPrice'])
      .groupby('AnoMes')['Receita'].sum()
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(receita_mensal.index.astype(str), receita_mensal.values, alpha=0.25, color=ACCENT)
ax.plot(receita_mensal.index.astype(str), receita_mensal.values,
        color=ACCENT, linewidth=2.5, marker='o', markersize=5)
ax.set_title('Receita Mensal — Pico de Vendas no Q4 (Nov–Dez)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Receita (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('output/02_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Os bundles identificados devem ser priorizados em campanhas de Q4 (Black Friday, Natal).')

In [ ]:
top_produtos = (
    df[df['Quantity'] > 0]
      .groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(20)
)

fig, ax = plt.subplots(figsize=(12, 6))
palette = [ACCENT if i == 0 else '#74B0FF' if i < 5 else '#ADB5BD' for i in range(20)]
sns.barplot(y=top_produtos.index, x=top_produtos.values, palette=palette, ax=ax)
ax.set_title('Top 20 Produtos por Unidades Vendidas', fontsize=13, fontweight='bold')
ax.set_xlabel('Total de Unidades Vendidas')
plt.tight_layout()
plt.savefig('output/03_top20_products.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Estes produtos estrela são os candidatos ideais como "produto âncora" nos bundles.')

---
## 2. Limpeza dos Dados

### 💼 Comentário Executivo
> Dados reais de ponto de venda contêm **ruído** que pode destruir a qualidade das regras de associação:
>
> | Problema | Por que importa |
> |----------|------------------|
> | Devoluções (InvoiceNo = 'C…') | Ensinaria o modelo que devolver A implica devolver B |
> | Qty / Preço ≤ 0 | Ajustes contábeis, não compras reais |
> | CustomerID nulo | Transações anônimas — não rastreáveis |
> | Descrição nula | Produto não identificado — sem insight de produto |
>
> **Decisão de negócio:** Mantemos apenas transações do UK **válidas e positivas** que representam comportamento real de compra.

In [ ]:
print(f'Linhas antes da limpeza: {len(df):,}')
df_clean = df.copy()

mask_dev = df_clean['InvoiceNo'].astype(str).str.startswith('C')
df_clean = df_clean[~mask_dev]
print(f'  └─ Removidas {mask_dev.sum():,} linhas de devoluções')

mask_qty = df_clean['Quantity'] <= 0
df_clean = df_clean[~mask_qty]
print(f'  └─ Removidas {mask_qty.sum():,} linhas com Quantity ≤ 0')

mask_preco = df_clean['UnitPrice'] <= 0
df_clean = df_clean[~mask_preco]
print(f'  └─ Removidas {mask_preco.sum():,} linhas com UnitPrice ≤ 0')

n_antes = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID', 'Description'])
print(f'  └─ Removidas {n_antes - len(df_clean):,} linhas com CustomerID ou Descrição nulos')

df_uk = df_clean[df_clean['Country'] == TARGET_COUNTRY].copy()
df_uk['Description'] = df_uk['Description'].str.strip()
print(f'  └─ Filtrado para {TARGET_COUNTRY}: {len(df_uk):,} linhas')

print(f'\n✅ Dataset limpo: {len(df_uk):,} linhas — {len(df_uk)/len(df)*100:.1f}% do original')
print(f'   Pedidos únicos: {df_uk["InvoiceNo"].nunique():,} | Clientes: {df_uk["CustomerID"].nunique():,} | Produtos: {df_uk["Description"].nunique():,}')

In [ ]:
etapas = [
    ('Original', len(df)),
    ('Sem devoluções', len(df[~df['InvoiceNo'].astype(str).str.startswith('C')])),
    ('Sem qty/preço neg.', len(df[~df['InvoiceNo'].astype(str).str.startswith('C') & (df['Quantity']>0) & (df['UnitPrice']>0)])),
    ('Sem nulos', len(df_clean)),
    ('Apenas UK', len(df_uk))
]
summary = pd.DataFrame(etapas, columns=['Etapa', 'Linhas'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(summary['Etapa'], summary['Linhas'], color=['#ADB5BD','#74B0FF','#4895EF','#4361EE','#3A0CA3'])
for i, (_, row) in enumerate(summary.iterrows()):
    ax.text(row['Linhas'] + 3000, i, f"{row['Linhas']:,}", va='center', fontsize=10)
ax.set_title('Impacto do Pipeline de Limpeza', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de linhas')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig('output/05_cleaning_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Transformação para Formato de Cesta (Matriz Binária de Transações)

### 💼 Comentário Executivo
> O Algoritmo Apriori requer uma **matriz binária de transações**: linhas = notas fiscais, colunas = produtos, `1` = comprado.
>
> Esta é a etapa mais cara computacionalmente. Um dataset de 500K linhas com 4.000 produtos gera uma matriz de 25K × 4K. Usamos o `TransactionEncoder` do mlxtend para uma representação esparsa e eficiente.
>
> **Decisão de design:** A unidade de análise é a **nota fiscal (cesta de compras)**, não o cliente — capturando o que é genuinamente comprado junto em uma única visita.

In [ ]:
lista_cestas = df_uk.groupby('InvoiceNo')['Description'].apply(list).tolist()
print(f'Total de cestas (notas fiscais): {len(lista_cestas):,}')
print(f'Exemplo de cesta: {lista_cestas[0][:5]} ...')

te = TransactionEncoder()
basket_df = pd.DataFrame(te.fit_transform(lista_cestas), columns=te.columns_)
print(f'\n✅ Matriz de cestas: {basket_df.shape[0]:,} notas × {basket_df.shape[1]:,} produtos')
print(f'   Densidade: {basket_df.values.mean()*100:.2f}% de 1s (muito esparsa — normal em varejo)')

In [ ]:
tam_cestas = df_uk.groupby('InvoiceNo')['Description'].nunique()

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(tam_cestas[tam_cestas <= 30], bins=30, color=ACCENT, edgecolor='white', alpha=0.9)
ax.axvline(tam_cestas.median(), color='#F72585', linestyle='--', linewidth=2,
           label=f'Mediana: {tam_cestas.median():.0f} itens')
ax.axvline(tam_cestas.mean(), color='#FF9F1C', linestyle='--', linewidth=2,
           label=f'Média: {tam_cestas.mean():.1f} itens')
ax.set_title('Distribuição do Tamanho da Cesta (produtos únicos por nota fiscal)', fontsize=13, fontweight='bold')
ax.set_xlabel('Nº de produtos únicos na cesta')
ax.legend()
plt.tight_layout()
plt.savefig('output/06_basket_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📌 {(tam_cestas == 1).mean()*100:.1f}% das cestas têm apenas 1 item — maior potencial de upsell via bundles.')

---
## 4. Algoritmo Apriori — Conjuntos de Itens Frequentes

### 💼 Comentário Executivo
> O **Algoritmo Apriori** é o padrão da indústria para Análise de Cesta de Compras.
>
> | Parâmetro | Valor | Significado de Negócio |
> |-----------|-------|------------------------|
> | **Suporte** | ≥ 1% | O conjunto aparece em pelo menos 1% dos pedidos |
> | **Confiança** | ≥ 20% | Dado que A foi comprado, B também foi comprado ≥20% das vezes |
> | **Lift** | > 1.0 | A associação é mais provável do que o acaso |
>
> **Intuição do Lift:** Lift = 3 significa que clientes que compram A têm **3× mais probabilidade** de comprar B do que a média. Este é o KPI principal para priorizar recomendações de cross-selling.

In [ ]:
print(f'⏳ Executando Apriori (suporte_mínimo={MIN_SUPPORT})...')
frequent_items = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
frequent_items['tamanho'] = frequent_items['itemsets'].apply(len)

print(f'✅ Conjuntos frequentes encontrados: {len(frequent_items):,}')
print('\nDistribuição por tamanho:')
print(frequent_items['tamanho'].value_counts().sort_index().to_string())

fi_export = frequent_items.copy()
fi_export['itemsets'] = fi_export['itemsets'].apply(lambda x: ', '.join(list(x)))
fi_export.to_csv(OUTPUT_ITEMS_CSV, index=False)
print(f'\n✅ Conjuntos exportados → {OUTPUT_ITEMS_CSV}')

In [ ]:
top_pares = (
    frequent_items[frequent_items['tamanho'] >= 2]
    .sort_values('support', ascending=False).head(15).copy()
)
top_pares['label'] = top_pares['itemsets'].apply(lambda x: ' + '.join(list(x)))

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_pares['label'], top_pares['support'],
        color=top_pares['tamanho'].map({2: ACCENT, 3: '#F72585'}))
ax.set_title('Top 15 Conjuntos de Itens Mais Frequentes (Pares e Triplas)', fontsize=13, fontweight='bold')
ax.set_xlabel('Suporte')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=ACCENT, label='Par (2 itens)'),
                   Patch(facecolor='#F72585', label='Tripla (3 itens)')])
plt.tight_layout()
plt.savefig('output/07_top_itemsets.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Regras de Associação — Top 20 por Lift

### 💼 Comentário Executivo
> As **regras de associação** têm o formato `{A} → {B}`: *clientes que compraram A também compraram B X vezes mais do que a média.*
>
> | Métrica | Caso de Uso |
> |---------|-------------|
> | **Lift > 5** | Recomendações personalizadas no site |
> | **Lift 3–5** | Bundle com desconto explícito |
> | **Confiança > 50%** | Gatilho automático no checkout |
>
> As regras são classificadas por **Lift** — a métrica mais informativa pois mede a surpresa genuína da co-compra, corrigindo pelo acaso.

In [ ]:
rules = association_rules(frequent_items, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules[rules['lift'] >= MIN_LIFT].copy()
rules['antecedentes'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequentes'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

print(f'✅ Regras geradas: {len(rules):,}')
for t in [1, 3, 5]:
    print(f'   Lift > {t}: {(rules["lift"] > t).sum():,}')

top20_rules = rules.sort_values('lift', ascending=False).head(20).reset_index(drop=True)

top20_disp = top20_rules[['antecedentes','consequentes','support','confidence','lift','leverage','conviction']].copy()
top20_disp.columns = ['Se compra →','Também compra →','Suporte','Confiança','Lift','Leverage','Convicção']
top20_disp[['Suporte','Confiança']] = top20_disp[['Suporte','Confiança']].applymap(lambda x: f'{x:.2%}')
top20_disp[['Lift','Leverage','Convicção']] = top20_disp[['Lift','Leverage','Convicção']].round(2)

print('\n🏆 TOP 20 REGRAS DE ASSOCIAÇÃO — Classificadas por Lift\n')
display(top20_disp)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
scatter = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'],
                     cmap='plasma', alpha=0.7, s=rules['lift']*8, edgecolors='white', linewidth=0.3)
plt.colorbar(scatter, ax=ax, label='Lift')
ax.set_xlabel('Suporte (frequência do par)', fontsize=11)
ax.set_ylabel('Confiança (Prob. de B dado A)', fontsize=11)
ax.set_title('Mapa de Regras: Suporte vs Confiança (tamanho e cor = Lift)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
for i, row in top20_rules.head(5).iterrows():
    ax.annotate(f'#{i+1} Lift={row["lift"]:.1f}', xy=(row['support'], row['confidence']),
                xytext=(5,5), textcoords='offset points', fontsize=7.5, color='#3A0CA3', fontweight='bold')
plt.tight_layout()
plt.savefig('output/08_rules_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📌 Regras no quadrante superior-direito = escaláveis + confiáveis + surpreendentes.')

In [ ]:
top_ant = top20_rules['antecedentes'].unique()[:10]
top_con = top20_rules['consequentes'].unique()[:10]
pivot_data = (
    top20_rules[top20_rules['antecedentes'].isin(top_ant) & top20_rules['consequentes'].isin(top_con)]
    .pivot_table(index='antecedentes', columns='consequentes', values='lift', aggfunc='max').fillna(0)
)
if not pivot_data.empty:
    fig, ax = plt.subplots(figsize=(13, 7))
    sns.heatmap(pivot_data, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax, cbar_kws={'label': 'Lift'})
    ax.set_title('Mapa de Calor do Lift — Top Regras de Associação', fontsize=13, fontweight='bold')
    ax.set_xlabel('Consequente ("Também compra")')
    ax.set_ylabel('Antecedente ("Se compra")')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig('output/09_lift_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
top20_plot = top20_rules.copy()
top20_plot['label_regra'] = top20_plot['antecedentes'].str[:25] + ' →\n' + top20_plot['consequentes'].str[:25]

fig, ax = plt.subplots(figsize=(12, 9))
cmap_vals = top20_plot['lift'].values
colors = plt.cm.plasma(plt.Normalize(cmap_vals.min(), cmap_vals.max())(cmap_vals[::-1]))
ax.barh(top20_plot['label_regra'][::-1], top20_plot['lift'][::-1], color=colors, edgecolor='white', linewidth=0.5)
for i, (_, row) in enumerate(top20_plot[::-1].iterrows()):
    ax.text(row['lift'] + 0.05, i, f'{row["lift"]:.2f}', va='center', fontsize=8.5)
ax.axvline(1, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Lift = 1 (acaso)')
ax.set_title('Top 20 Regras de Associação — Classificadas por Lift', fontsize=13, fontweight='bold')
ax.set_xlabel('Lift')
ax.legend()
plt.tight_layout()
plt.savefig('output/10_top20_rules_lift.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Exportação dos Resultados → Power BI

### 💼 Comentário Executivo
> Quatro arquivos CSV são exportados para conectar esta análise a um dashboard no Power BI:
>
> | Arquivo | Descrição | Uso no Power BI |
> |---------|-----------|------------------|
> | `association_rules.csv` | Todas as regras com métricas completas | Tabela principal com slicer |
> | `frequent_items.csv` | Conjuntos frequentes com suporte | Gráfico de bolhas por produto |
> | `kpis_summary.csv` | KPIs do negócio | Cartões de métricas |
> | `top20_rules.csv` | Top 20 regras para ação imediata | Tabela de recomendações |
>
> **No Power BI:** Importe com *Obter Dados → Texto/CSV*. Use `antecedent` como segmentação e crie medida DAX: `Regras Lift Alto = CALCULATE(COUNTROWS(rules), rules[lift] >= 3)`

In [ ]:
# Todas as regras
rules_export = rules[['antecedentes','consequentes','support','confidence','lift','leverage','conviction','zhangs_metric']].copy()
rules_export.columns = ['antecedent','consequent','support','confidence','lift','leverage','conviction','zhangs_metric']
rules_export = rules_export.sort_values('lift', ascending=False).reset_index(drop=True)
rules_export['rank_regra'] = rules_export.index + 1
rules_export['nivel_lift'] = pd.cut(rules_export['lift'], bins=[0,2,3,5,float('inf')],
                                     labels=['Baixo (1-2)','Médio (2-3)','Alto (3-5)','Muito Alto (>5)'])
rules_export.to_csv(OUTPUT_RULES_CSV, index=False)
print(f'✅ {len(rules_export):,} regras → {OUTPUT_RULES_CSV}')

# Top 20
top20_exp = top20_rules[['antecedentes','consequentes','support','confidence','lift']].copy()
top20_exp.columns = ['antecedent','consequent','support','confidence','lift']
top20_exp['rank'] = range(1, 21)
top20_exp.to_csv('output/top20_rules.csv', index=False)
print('✅ Top 20 regras → output/top20_rules.csv')

# KPIs
receita_total = (df_uk['Quantity'] * df_uk['UnitPrice']).sum()
pedidos_total = df_uk['InvoiceNo'].nunique()
kpis = pd.DataFrame({
    'kpi': ['Receita Total (£)','Total de Pedidos','Ticket Médio/AOV (£)','Total de Clientes',
            'Produtos Únicos','Total de Regras','Regras Lift > 3','Lift Máximo'],
    'valor': [
        round(receita_total,2), pedidos_total, round(receita_total/pedidos_total,2),
        df_uk['CustomerID'].nunique(), df_uk['Description'].nunique(),
        len(rules_export), int((rules_export['lift']>3).sum()),
        round(rules_export['lift'].max(),2)
    ]
})
kpis.to_csv('output/kpis_summary.csv', index=False)
print('✅ KPIs → output/kpis_summary.csv')

print('\n' + '='*52)
print('📊 RESUMO EXECUTIVO DO PROJETO')
print('='*52)
for _, row in kpis.iterrows():
    val = f'{row["valor"]:,.2f}' if isinstance(row['valor'], float) else f'{row["valor"]:,}'
    print(f'  {row["kpi"]:<30} {val}')

In [ ]:
print('📁 Arquivos gerados em /output:')
for f in [OUTPUT_RULES_CSV, OUTPUT_ITEMS_CSV, 'output/top20_rules.csv', 'output/kpis_summary.csv']:
    if os.path.exists(f):
        print(f'  ✅ {f} ({os.path.getsize(f)/1024:.1f} KB)')
    else:
        print(f'  ❌ {f} — NÃO ENCONTRADO')

imgs = [f for f in os.listdir('output') if f.endswith('.png')]
print(f'\n🖼️  Gráficos gerados: {len(imgs)}')
print('\n🚀 Análise concluída. Todos os arquivos prontos para importação no Power BI.')

---
## 📋 Resumo para Entrevista

### Qual problema resolvemos?
Uma loja online do Reino Unido queria aumentar o **AOV** identificando produtos comprados juntos para criar bundles e estratégias de cross-selling baseadas em dados.

### Pipeline
1. **AED** — UK = ~80% da receita; Q4 é a janela crítica de vendas.
2. **Limpeza** — Pipeline reprodutível: remoção de devoluções → valores negativos → nulos → não-UK.
3. **Matriz de Cesta** — 500K+ linhas → matriz binária nota × produto via `TransactionEncoder`.
4. **Apriori** — Conjuntos frequentes com suporte ≥ 1%, máximo triplas para interpretabilidade.
5. **Regras de Associação** — Classificadas por Lift; confiança ≥ 20%.
6. **Exportação** — 4 CSVs + 10 gráficos PNG prontos para Power BI.

### Impacto de Negócio
| Cenário | Ação | Impacto Esperado |
|---------|------|------------------|
| Lift > 5 | Motor de recomendações personalizado | +8–15% CTR |
| Lift 3–5 | Bundle com desconto de 10% | +12–20% AOV |
| Confiança > 50% | Gatilho automático no checkout | +5–10% Conversão |

### Decisões Técnicas Principais
- **Apriori vs FP-Growth:** Apriori escolhido pela interpretabilidade; FP-Growth é preferível para >5M linhas.
- **Lift como métrica principal:** Suporte/Confiança podem ser altos por acaso; Lift mede a associação genuína.
- **Filtro apenas UK:** Comportamento de compra varia por mercado — misturar países dilui regras específicas.

---
*Todos os arquivos em `/output/` estão prontos para importação direta no Power BI.*